# Objective Quest DATAQUEST 5.0 — V14 Xiaomi-TabLDM Full-Target Ordinal Regression (Runtime-Safe Revision)

**Frozen research contract.** V14 tests the official **Xiaomi-TabLDM regressor** (Git commit `b8ec5c87b66434d941f701795584d86853fd7f29`, HF revision `0e10427273bc862fb60fe7df84c89e5298e16cc7`) as a genuinely new full-target tabular-foundation-model regime. The target remains numeric `1..4`; QWK is produced only after leakage-safe ordinal threshold calibration.

Key constraints:
- competition-provided respondent features only; no external respondent rows, labels, or pseudo-ground-truth;
- one modeling row per unique respondent `id`; duplicate raw rows are audited, never allowed to cross folds;
- `id` and `household_id` are excluded from model features;
- low-cardinality survey codes are explicitly represented as pandas categorical columns so TabLDM does not interpret nominal survey codes as magnitudes;
- audited continuous/ordinal fields remain numeric;
- frozen Recovery (`Public LB 0.48334`) is the scientific control, reconstructed only from the supplied historical prediction artifact bundle;
- **Stage R** proves full-context runtime with `n_estimators=1`;
- **Stage S** evaluates frozen folds `0` and `2` with `n_estimators=2`;
- full five-fold OOF/test inference uses `n_estimators=4` only after preregistered promotion;
- **query-row chunking is runtime-only**: TabLDM's `batch_size` batches ensemble estimators, not validation/test rows, so prediction is issued through repeated official `predict()` calls on bounded query chunks while keeping the same fitted context and ensemble;
- T4×2 is used through isolated fold workers when two GPUs are visible; if Kaggle exposes only one T4, the identical fold jobs run sequentially on that GPU. This changes only scheduling, never features, seed, estimator count, target, or validation; failed workers are retried in conservative runtime mode;
- all headline QWK values use out-of-fold thresholds. Global threshold fitting is never used as headline validation.

**Kaggle inputs required:** competition CSV files and the supplied `results-v1_v12.zip` (or extracted equivalent) containing historical V2/V5.1/V6.1/V7 prediction artifacts.


## 0. Resolve and install the official Xiaomi-TabLDM source at one exact Git commit


In [1]:
import sys, subprocess, json, re
from pathlib import Path

REPO_URL='https://github.com/xiaomi-research/xiaomi-tabldm.git'
INSTALL_RECORD=Path('/kaggle/working/v14_tabldm_install.json') if Path('/kaggle/working').exists() else Path('./v14_tabldm_install.json')

def sh(cmd):
    print('+', ' '.join(map(str,cmd)))
    return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT).strip()

# Exact official Xiaomi-TabLDM source revision frozen for reproducibility.
PINNED_COMMIT='b8ec5c87b66434d941f701795584d86853fd7f29'
if re.fullmatch(r'[0-9a-f]{40}',PINNED_COMMIT) is None:
    raise RuntimeError('Invalid pinned Xiaomi-TabLDM commit.')

subprocess.check_call([
    sys.executable,'-m','pip','install','-q','--disable-pip-version-check',
    f'git+{REPO_URL}@{PINNED_COMMIT}'
])
INSTALL_RECORD.write_text(json.dumps({'repo':REPO_URL,'resolved_commit':PINNED_COMMIT},indent=2))
print('Pinned Xiaomi-TabLDM commit:',PINNED_COMMIT)


Pinned Xiaomi-TabLDM commit: b8ec5c87b66434d941f701795584d86853fd7f29


## 1. Imports, hardware contract, frozen gates, and reproducibility


In [2]:
import os, gc, sys, json, math, time, random, shutil, hashlib, zipfile, traceback, platform, subprocess, inspect, importlib.metadata, re
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score, roc_auc_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import StratifiedGroupKFold

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ.setdefault('PYTHONHASHSEED',str(SEED))
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')

OUT=Path('/kaggle/working/oqdq5_v14') if Path('/kaggle/working').exists() else Path('./oqdq5_v14')
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True,exist_ok=True)

@dataclass(frozen=True)
class Cfg:
    target:str='life_satisfaction'; id_col:str='id'; household_col:str='household_id'
    n_folds:int=5; screen_folds:tuple=(0,2)
    expected_train_rows:int=95609; expected_train_ids:int=94663
    expected_test_rows:int=23902; expected_test_ids:int=23666
    recovery_public_lb:float=0.48334
    runtime_estimators:int=1; screen_estimators:int=2; full_estimators:int=4
    estimator_batch_size:int=1; n_jobs_per_worker:int=4
    # Runtime-only query batching. 2,048 rows was proven barely feasible (~98% VRAM) in the failed run;
    # use a smaller normal chunk and a still smaller conservative retry chunk.
    query_chunk_rows:int=1792; safe_query_chunk_rows:int=896; runtime_probe_rows:int=1792
    use_amp:str='auto'; offload_mode:str='auto'; kv_cache:bool=False
    checkpoint_repo:str='occams/Xiaomi-TabLDM'; checkpoint_revision:str='0e10427273bc862fb60fe7df84c89e5298e16cc7'; checkpoint_file:str='checkpoints/reg_default.ckpt'
    bootstrap_reps:int=2000
    # Stage-S gates inherited from the rigorous foundation-model screen.
    stage_auc_gain:float=0.005; stage_qwk_gain:float=0.002; stage_catastrophic_auc:float=-0.005
    stage_novel_qwk_floor:float=-0.0015; stage_novel_resid_corr:float=0.95; stage_novel_auc_floor:float=-0.001
    # Full-CV gates.
    scientific_qwk_gain:float=0.002; submission_qwk_gain:float=0.0015
    submission_boot_p:float=0.90; submission_min_positive_folds:int=4; submission_min_fold:float=-0.0015
    challenge_qwk_gain:float=0.0025; challenge_auc_gain:float=0.005
CFG=Cfg()

assert torch.cuda.is_available(), 'CUDA is required.'
N_VISIBLE_GPUS=int(torch.cuda.device_count())
assert N_VISIBLE_GPUS>=1, 'At least one CUDA GPU is required.'
VISIBLE_GPUS=tuple(range(N_VISIBLE_GPUS))
GPU_SCHEDULING_MODE='dual_parallel' if N_VISIBLE_GPUS>=2 else 'single_sequential'
for i in VISIBLE_GPUS:
    p=torch.cuda.get_device_properties(i)
    print(i,p.name,round(p.total_memory/2**30,2),'GiB',f'cc={p.major}.{p.minor}')
print('Visible GPUs',N_VISIBLE_GPUS,'scheduling',GPU_SCHEDULING_MODE)
if N_VISIBLE_GPUS<2:
    print('WARNING: Kaggle exposed only one GPU. V14 will preserve the exact experiment and run fold workers sequentially on GPU 0.')
print('OUT',OUT)
print('torch',torch.__version__,'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__,'scipy',scipy.__version__)


0 Tesla T4 14.56 GiB cc=7.5
1 Tesla T4 14.56 GiB cc=7.5
Visible GPUs 2 scheduling dual_parallel
OUT /kaggle/working/oqdq5_v14
torch 2.10.0+cu128 numpy 2.0.2 pandas 2.3.3 sklearn 1.6.1 scipy 1.16.3


## 2. Official package/checkpoint provenance and API integrity


In [3]:
from huggingface_hub import HfApi, hf_hub_download
from tabldm import TabLDMRegressor
import tabldm

install=json.loads((Path('/kaggle/working/v14_tabldm_install.json') if Path('/kaggle/working').exists() else Path('./v14_tabldm_install.json')).read_text())
EXPECTED_COMMIT=install['resolved_commit']

def installed_git_commit():
    d=importlib.metadata.distribution('Xiaomi-TabLDM')
    raw=d.read_text('direct_url.json')
    if not raw: raise RuntimeError('Installed TabLDM package has no direct_url.json provenance.')
    obj=json.loads(raw); commit=obj.get('vcs_info',{}).get('commit_id')
    if re.fullmatch(r'[0-9a-f]{40}',str(commit or '')) is None:
        raise RuntimeError(f'Invalid installed Git provenance: {obj}')
    return commit,obj

TABLDM_COMMIT,DIRECT_URL=installed_git_commit()
assert TABLDM_COMMIT==EXPECTED_COMMIT, (TABLDM_COMMIT,EXPECTED_COMMIT)

# Verify only Xiaomi-TabLDM's declared runtime requirements; do not fail on unrelated Kaggle package conflicts.
from packaging.requirements import Requirement
from packaging.version import Version
REQ_REPORT=[]
for raw in importlib.metadata.requires('Xiaomi-TabLDM') or []:
    req=Requirement(raw)
    if req.marker is not None and not req.marker.evaluate():
        continue
    try:
        ver=importlib.metadata.version(req.name)
    except importlib.metadata.PackageNotFoundError as e:
        raise RuntimeError(f'Missing Xiaomi-TabLDM dependency: {raw}') from e
    ok=(not req.specifier) or (Version(ver) in req.specifier)
    REQ_REPORT.append({'requirement':raw,'installed':ver,'ok':bool(ok)})
    if not ok: raise RuntimeError(f'Xiaomi-TabLDM dependency mismatch: {raw}; installed {ver}')

sig=inspect.signature(TabLDMRegressor.__init__).parameters
required=['n_estimators','batch_size','kv_cache','model_path','allow_auto_download','device','use_amp','use_fa3','offload_mode','disk_offload_dir','random_state','n_jobs','verbose','inference_config']
missing=[x for x in required if x not in sig]
if missing: raise RuntimeError(f'TabLDMRegressor API drift; missing parameters {missing}')

HF_REVISION=CFG.checkpoint_revision
if re.fullmatch(r'[0-9a-f]{40}',str(HF_REVISION or '')) is None:
    raise RuntimeError(f'Invalid frozen HF model revision: {HF_REVISION!r}')
CKPT=Path(hf_hub_download(CFG.checkpoint_repo,CFG.checkpoint_file,revision=HF_REVISION))

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
CKPT_SHA=sha256(CKPT)
REGRESSOR_SOURCE=Path(inspect.getfile(TabLDMRegressor)).resolve()
REGRESSOR_SHA=sha256(REGRESSOR_SOURCE)

PROVENANCE={
    'tabldm_commit':TABLDM_COMMIT,'tabldm_direct_url':DIRECT_URL,
    'tabldm_module':str(Path(tabldm.__file__).resolve()),'regressor_source_sha256':REGRESSOR_SHA,
    'checkpoint_repo':CFG.checkpoint_repo,'checkpoint_revision':HF_REVISION,
    'checkpoint_file':CFG.checkpoint_file,'checkpoint_local_path':str(CKPT),'checkpoint_sha256':CKPT_SHA,
    'package_version':getattr(tabldm,'__version__','unknown'),'dependency_requirements':REQ_REPORT,
}
json.dump(PROVENANCE,open(OUT/'foundation_provenance.json','w'),indent=2,default=str)
print(json.dumps(PROVENANCE,indent=2))


checkpoints/reg_default.ckpt:   0%|          | 0.00/569M [00:00<?, ?B/s]

{
  "tabldm_commit": "b8ec5c87b66434d941f701795584d86853fd7f29",
  "tabldm_direct_url": {
    "url": "https://github.com/xiaomi-research/xiaomi-tabldm.git",
    "vcs_info": {
      "commit_id": "b8ec5c87b66434d941f701795584d86853fd7f29",
      "requested_revision": "b8ec5c87b66434d941f701795584d86853fd7f29",
      "vcs": "git"
    }
  },
  "tabldm_module": "/usr/local/lib/python3.12/dist-packages/tabldm/__init__.py",
  "regressor_source_sha256": "ef397bb82eee55b4f3deef5099f2878db6b44298a33dbff1e71b539878eb7eaa",
  "checkpoint_repo": "occams/Xiaomi-TabLDM",
  "checkpoint_revision": "0e10427273bc862fb60fe7df84c89e5298e16cc7",
  "checkpoint_file": "checkpoints/reg_default.ckpt",
  "checkpoint_local_path": "/root/.cache/huggingface/hub/models--occams--Xiaomi-TabLDM/snapshots/0e10427273bc862fb60fe7df84c89e5298e16cc7/checkpoints/reg_default.ckpt",
  "checkpoint_sha256": "c2be1257f1f3bcfd01ee73fbf964db1e07a535e84fc3811736aadef571adcb48",
  "package_version": "0.1.0",
  "dependency_requirement

## 3. Data discovery, duplicate audit, frozen folds, and historical Recovery control


In [4]:
def roots():
    r=[Path('/kaggle/input'),Path('/mnt/data')]
    c=Path.cwd().resolve()
    if c!=Path('/') and not str(c).startswith('/proc'): r.append(c)
    return [x for x in r if x.exists()]

def locate_data():
    hits=[]
    for root in roots():
        for p in root.rglob('train_final.csv'):
            d=p.parent
            if (d/'test_final.csv').exists() and (d/'data_dictionary.csv').exists(): hits.append(d.resolve())
    hits=list(dict.fromkeys(hits))
    if not hits: raise FileNotFoundError('train_final.csv/test_final.csv/data_dictionary.csv not found under Kaggle inputs.')
    hits.sort(key=lambda p:(not (p/'sample_submission.csv').exists(),len(str(p))))
    return hits[0]

class Store:
    def __init__(self,root=None,z=None):
        self.root=Path(root) if root else None; self.z=Path(z) if z else None
        self.zf=zipfile.ZipFile(self.z) if self.z else None
    def csv(self,rel,**kw):
        if self.root: return pd.read_csv(self.root/rel,**kw)
        with self.zf.open(rel) as f: return pd.read_csv(f,**kw)
    def source(self): return str(self.root or self.z)
    def close(self):
        if self.zf: self.zf.close()

def locate_store():
    marker='results/v2/oof_v2.csv'; dirs=[]; zs=[]
    for root in roots():
        for p in root.rglob('oof_v2.csv'):
            if p.as_posix().endswith('/results/v2/oof_v2.csv'): dirs.append(p.parents[2].resolve())
        for z in root.rglob('*.zip'):
            try:
                with zipfile.ZipFile(z) as q:
                    if marker in q.namelist(): zs.append(z.resolve())
            except zipfile.BadZipFile: pass
    dirs=list(dict.fromkeys(dirs)); zs=list(dict.fromkeys(zs))
    if dirs: return Store(root=sorted(dirs,key=lambda p:len(str(p)))[0])
    if len(zs)==1: return Store(z=zs[0])
    named=[z for z in zs if z.name in {'results-v1_v12.zip','results.zip'}]
    if len(named)==1: return Store(z=named[0])
    if not zs: raise FileNotFoundError('Historical results-v1_v12.zip is required for the frozen Recovery control and fold contract.')
    raise RuntimeError(f'Ambiguous historical result archives: {zs}')

DATA=locate_data(); STORE=locate_store()
train=pd.read_csv(DATA/'train_final.csv',low_memory=False)
test=pd.read_csv(DATA/'test_final.csv',low_memory=False)
dd=pd.read_csv(DATA/'data_dictionary.csv')
assert len(train)==CFG.expected_train_rows and train.id.nunique()==CFG.expected_train_ids
assert len(test)==CFG.expected_test_rows and test.id.nunique()==CFG.expected_test_ids
assert CFG.target in train and CFG.target not in test

# Exact duplicates are allowed only if every non-ID field is identical.
for df,name in [(train,'train'),(test,'test')]:
    dup=df[df.duplicated(CFG.id_col,False)]
    if len(dup):
        g=dup.groupby(CFG.id_col,sort=False)
        bad=[c for c in df.columns if c!=CFG.id_col and g[c].nunique(dropna=False).max()>1]
        if bad: raise RuntimeError(f'{name} duplicate IDs disagree in columns: {bad[:10]}')

# Frozen V2 primary-fold assignment is the source of truth.
v2meta=STORE.csv('results/v2/oof_v2.csv',usecols=['id','life_satisfaction','primary_fold'])
canon=v2meta.drop_duplicates('id',keep='first').reset_index(drop=True)
assert len(canon)==CFG.expected_train_ids and set(canon.primary_fold.astype(int))==set(range(5))
train_uid=canon[['id']].merge(train.drop_duplicates('id',keep='first'),on='id',validate='one_to_one')
train_uid['__fold__']=canon.primary_fold.astype(np.int8).to_numpy()
test_uid=test.drop_duplicates('id',keep='first').reset_index(drop=True)
assert np.array_equal(train_uid.life_satisfaction.astype(int),canon.life_satisfaction.astype(int))
assert set(train_uid.id).isdisjoint(set(test_uid.id))

COMP={
'v2_final':('v2','oof_v2.csv','test_predictions_v2.csv','oof_final_continuous','test_final_continuous'),
'v5_final':('v5_1','oof_v5.csv','test_predictions_v5.csv','oof_final_honest_continuous','pred_final_continuous'),
'v61_final':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_final_crossfit','pred_final_continuous'),
'v7_stack':('v7','oof_v7.csv','test_scores_v7.csv','v7_stack','v7_stack')}
META={'v2':('primary_fold','life_satisfaction'),'v5_1':('fold','life_satisfaction'),'v6_1':('primary_fold','life_satisfaction'),'v7':('primary_fold','life_satisfaction')}
train_ids=train_uid.id.to_numpy(); test_ids=test_uid.id.to_numpy(); y=train_uid.life_satisfaction.astype(int).to_numpy(); folds=train_uid.__fold__.astype(int).to_numpy()

def test_ids_for(df):
    if 'id' in df: return df.id
    if 'test_feature_id' in df: return df.test_feature_id
    if 'row_index' in df:
        if not np.array_equal(df.row_index.to_numpy(),np.arange(len(test))): raise RuntimeError('Historical row_index mapping drift.')
        return test.id.reset_index(drop=True)
    raise KeyError('No historical test ID mapping.')

def load_component(spec):
    v,of,tf,oc,tc=spec; od=STORE.csv(f'results/{v}/{of}'); td=STORE.csv(f'results/{v}/{tf}')
    fc,yc=META[v]; g=od[['id',yc,fc,oc]].groupby('id',sort=False)
    assert g[yc].nunique().max()==1 and g[fc].nunique().max()==1 and (g[oc].max()-g[oc].min()).max()<1e-8
    a=g.agg({yc:'first',oc:'mean'}).reset_index(); a=canon[['id']].merge(a,on='id',validate='one_to_one')
    assert np.array_equal(a[yc].astype(int),y)
    td=td.copy(); td['_id']=test_ids_for(td).to_numpy(); h=td[['_id',tc]].groupby('_id',sort=False)
    assert (h[tc].max()-h[tc].min()).max()<1e-8
    b=h[tc].mean().reset_index().rename(columns={'_id':'id'}); b=test_uid[['id']].merge(b,on='id',validate='one_to_one')
    return a[oc].to_numpy(float),b[tc].to_numpy(float)

arr={n:load_component(s) for n,s in COMP.items()}
REC_NAMES=('v2_final','v5_final','v61_final','v7_stack')
RW={0:(0.6131481145225551,0.1984633896432871,0.18838849583415773,0.0),1:(0.5736087705637004,0.2130793852900037,0.20117728057551384,0.012134563570782105),2:(0.5260926035320435,0.22651646669428574,0.20045779567769734,0.04693313409597358),3:(0.412997730762594,0.12147998345945334,0.2995554634343377,0.16596682234361498),4:(0.4419367210113302,0.27031416820745807,0.2416404622741877,0.04610864850702414)}
RT={0:(1.5694444444444446,2.011875,2.4941666666666666),1:(1.57,2.011875,2.5100000000000002),2:(1.57,2.0162500000000003,2.5100000000000002),3:(1.5683333333333334,2.011875,2.524166666666667),4:(1.568888888888889,2.014375,2.5100000000000002)}
RFW=np.array((0.5333147962860486,0.2160045364556684,0.20393903979487613,0.0467416274634069)); RFT=np.array((1.5694444444444446,2.011875,2.5100000000000002))
RX=np.column_stack([arr[n][0] for n in REC_NAMES]); RTX=np.column_stack([arr[n][1] for n in REC_NAMES])
recovery_oof=np.empty(len(y)); recovery_class=np.empty(len(y),int)
for f in range(5):
    m=folds==f; recovery_oof[m]=RX[m]@np.array(RW[f]); t=np.array(RT[f]); recovery_class[m]=1+(recovery_oof[m]>=t[0])+(recovery_oof[m]>=t[1])+(recovery_oof[m]>=t[2])
recovery_test=RTX@RFW; recovery_test_class=1+(recovery_test>=RFT[0])+(recovery_test>=RFT[1])+(recovery_test>=RFT[2])
RECOVERY_QWK=float(cohen_kappa_score(y,recovery_class,weights='quadratic'))
assert abs(RECOVERY_QWK-0.48186721909562624)<1e-12

# Always export the exact known champion as a control artifact.
control_sub=pd.DataFrame({'id':test_ids,'life_satisfaction':recovery_test_class.astype(int)})
assert len(control_sub)==CFG.expected_test_ids and control_sub.id.is_unique
control_sub.to_csv(OUT/'submission_v14_recovery_control.csv',index=False)
print('DATA',DATA,'HISTORY',STORE.source())
print('Unique respondents',len(train_uid),len(test_uid),'Frozen Recovery OOF',RECOVERY_QWK,'Public LB',CFG.recovery_public_lb)


DATA /kaggle/input/competitions/oq-dq-5 HISTORY /kaggle/input/datasets/riantamaputra/dataquest-2026-historical-results
Unique respondents 94663 23666 Frozen Recovery OOF 0.48186721909562624 Public LB 0.48334


## 4. Survey-native feature typing for Xiaomi-TabLDM


In [5]:
FEATURES=[c for c in test.columns if c not in {CFG.id_col,CFG.household_col}]
# Frozen semantic typing from the audited survey contract. Only these continuous fields are treated as true magnitudes.
CONT=[c for c in ['income_to_poverty_ratio','hours_worked_per_week','survey_weight','days_missed_work','height_inches','age','weight_pounds'] if c in FEATURES]
ORD=[c for c in ['education_level','max_education_in_family','income_poverty_category','food_security_4cat','food_security_3cat','general_health_status','depression_frequency','anxiety_frequency','anxiety_level','survey_year','data_year'] if c in FEATURES]
NOM=[c for c in FEATURES if c not in set(CONT+ORD)]
assert CFG.target not in FEATURES and CFG.id_col not in FEATURES and CFG.household_col not in FEATURES
assert len(FEATURES)==len(CONT)+len(ORD)+len(NOM)

SEMANTIC={'features':FEATURES,'continuous':CONT,'ordinal_numeric':ORD,'nominal_categorical':NOM,
          'policy':'raw competition survey view; no target-driven feature selection; nominal survey codes are pandas categorical'}
json.dump(SEMANTIC,open(OUT/'semantic_contract_v14.json','w'),indent=2)

# Workers consume these exact unique-row tables and recreate categorical dtypes locally.
train_uid.to_csv(OUT/'train_unique_v14.csv',index=False)
test_uid.to_csv(OUT/'test_unique_v14.csv',index=False)
print('Feature count',len(FEATURES),'continuous',len(CONT),'ordinal numeric',len(ORD),'nominal categorical',len(NOM))


Feature count 176 continuous 7 ordinal numeric 11 nominal categorical 158


## 5. Isolated one-GPU workers with dynamic 1×/2× T4 orchestration

In [6]:
WORKER_SOURCE=r'''#!/usr/bin/env python
import os,sys,json,time,gc,traceback,argparse,random,hashlib,shutil,resource
from pathlib import Path
import numpy as np,pandas as pd,torch
from tabldm import TabLDMRegressor

ap=argparse.ArgumentParser()
ap.add_argument('--out',required=True)
ap.add_argument('--fold',type=int,required=True)
ap.add_argument('--stage',required=True)
ap.add_argument('--n-estimators',type=int,required=True)
ap.add_argument('--query-chunk-rows',type=int,required=True)
ap.add_argument('--predict-test',action='store_true')
ap.add_argument('--runtime-probe',action='store_true')
ap.add_argument('--safe-mode',action='store_true')
a=ap.parse_args()

OUT=Path(a.out); job=OUT/f'{a.stage}_fold{a.fold}'; job.mkdir(parents=True,exist_ok=True)
SEED=14000+a.fold*101+a.n_estimators*17
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(int(os.environ.get('TABLDM_CPU_THREADS','4')))

cfg=json.load(open(OUT/'worker_contract.json')); sem=json.load(open(OUT/'semantic_contract_v14.json'))
features=sem['features']; nominal=sem['nominal_categorical']

# Do not materialize the test table in screen/runtime workers.
train_cols=list(dict.fromkeys([cfg['id_col'],cfg['target'],'__fold__']+features))
tr=pd.read_csv(OUT/'train_unique_v14.csv',usecols=train_cols,low_memory=False)
folds=tr['__fold__'].astype(int).to_numpy()
y=tr[cfg['target']].astype(float).to_numpy()
train_mask=folds!=a.fold; valid_mask=folds==a.fold

def view(df):
    X=df[features].copy()
    for c in nominal:
        X[c]=X[c].astype('category')
    return X

A=view(tr.loc[train_mask])
valid_raw=tr.loc[valid_mask,[cfg['id_col']]+features].reset_index(drop=True)
if a.runtime_probe:
    valid_raw=valid_raw.iloc[:min(len(valid_raw),int(cfg['runtime_probe_rows']))].copy()

te=None
if a.predict_test:
    test_cols=list(dict.fromkeys([cfg['id_col']]+features))
    te=pd.read_csv(OUT/'test_unique_v14.csv',usecols=test_cols,low_memory=False)

off=Path(cfg['offload_root'])/f'{a.stage}_fold{a.fold}'
shutil.rmtree(off,ignore_errors=True); off.mkdir(parents=True,exist_ok=True)

# Conservative mode changes only inference storage/batching policy, never the model/data/seed.
inference_config=None
if a.safe_mode:
    common={
        'safety_factor':0.60,
        'offload':'disk',
        'disk_offload_dir':str(off),
        'disk_min_free_mb':1024.0,
        'disk_flush_mb':1024.0,
        'disk_cleanup':True,
        'use_async':False,
        'async_depth':1,
        'max_pinned_memory_mb':512.0,
    }
    inference_config={
        'COL_CONFIG':dict(common),
        'ROW_CONFIG':dict(common),
        'ICL_CONFIG':dict(common),
    }

def peak_rss_gib():
    # Linux ru_maxrss is KiB.
    return float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024/1024)

def chunked_predict(model,raw_df,chunk_rows,label):
    n=len(raw_df)
    if n==0:
        return np.empty(0,dtype=np.float64),0
    out=[]; calls=0
    for lo in range(0,n,chunk_rows):
        hi=min(n,lo+chunk_rows)
        Xq=view(raw_df.iloc[lo:hi])
        q=np.asarray(model.predict(Xq),dtype=np.float64).reshape(-1)
        if len(q)!=(hi-lo) or not np.isfinite(q).all():
            raise RuntimeError(f'Invalid {label} predictions for rows [{lo}:{hi})')
        out.append(q); calls+=1
        del Xq,q
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if calls%4==0:
            gc.collect()
    p=np.concatenate(out)
    if len(p)!=n or not np.isfinite(p).all():
        raise RuntimeError(f'Invalid concatenated {label} predictions')
    return p,calls

started=time.time(); model=None
try:
    model=TabLDMRegressor(
        n_estimators=a.n_estimators,
        batch_size=cfg['batch_size'],
        kv_cache=False,
        model_path=cfg['checkpoint'],allow_auto_download=False,
        device='cuda',use_amp=cfg['use_amp'],use_fa3=False,
        offload_mode=cfg['offload_mode'],disk_offload_dir=str(off),
        random_state=SEED,n_jobs=int(cfg['n_jobs']),verbose=bool(a.runtime_probe),
        inference_config=inference_config,
    )
    model.fit(A,y[train_mask])
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    p,nv_calls=chunked_predict(model,valid_raw,int(a.query_chunk_rows),'validation')
    np.savez_compressed(
        job/'valid_predictions.npz',
        id=valid_raw[cfg['id_col']].to_numpy(),
        pred=p
    )

    nt_calls=0
    if a.predict_test:
        pt,nt_calls=chunked_predict(model,te,int(a.query_chunk_rows),'test')
        np.savez_compressed(
            job/'test_predictions.npz',
            id=te[cfg['id_col']].to_numpy(),
            pred=pt
        )

    total_gib=float(torch.cuda.get_device_properties(0).total_memory/2**30)
    max_alloc=float(torch.cuda.max_memory_allocated()/2**30)
    rec={
        'status':'success','stage':a.stage,'fold':a.fold,'seed':SEED,
        'n_estimators':a.n_estimators,'train_rows':int(train_mask.sum()),
        'valid_rows_predicted':int(len(valid_raw)),
        'test_rows_predicted':int(len(te) if a.predict_test else 0),
        'query_chunk_rows':int(a.query_chunk_rows),
        'valid_predict_calls':int(nv_calls),'test_predict_calls':int(nt_calls),
        'safe_mode':bool(a.safe_mode),
        'features':int(len(features)),'nominal_features':int(len(nominal)),
        'runtime_sec':float(time.time()-started),
        'cuda_name':torch.cuda.get_device_name(0),
        'cuda_total_gib':total_gib,
        'max_cuda_alloc_gib':max_alloc,
        'cuda_peak_fraction':float(max_alloc/total_gib),
        'peak_rss_gib':peak_rss_gib(),
        'offload_dir':str(off)
    }
    json.dump(rec,open(job/'runtime.json','w'),indent=2)
    print(json.dumps(rec))
except Exception as e:
    rec={
        'status':'failure','stage':a.stage,'fold':a.fold,
        'n_estimators':a.n_estimators,'query_chunk_rows':int(a.query_chunk_rows),
        'safe_mode':bool(a.safe_mode),
        'runtime_sec':float(time.time()-started),
        'peak_rss_gib':peak_rss_gib(),
        'error':repr(e),'traceback':traceback.format_exc()
    }
    json.dump(rec,open(job/'failure.json','w'),indent=2)
    print(json.dumps(rec),file=sys.stderr)
    raise
finally:
    if model is not None: del model
    del A,valid_raw,tr
    if te is not None: del te
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
'''
WORKER=OUT/'tabldm_worker_v14.py'; WORKER.write_text(WORKER_SOURCE)

# Pick the writable filesystem with the most free space for automatic CPU/disk offloading.
def choose_offload_root():
    rows=[]
    for p in [Path('/kaggle/temp'),Path('/kaggle/working'),OUT]:
        try:
            p.mkdir(parents=True,exist_ok=True); rows.append((shutil.disk_usage(p).free,p))
        except Exception: pass
    if not rows: raise RuntimeError('No writable offload directory.')
    free,p=max(rows,key=lambda z:z[0])
    d=p/'oqdq5_v14_tabldm_offload'; d.mkdir(parents=True,exist_ok=True)
    print('Offload root',d,'free GiB',round(free/2**30,2))
    return d

OFFLOAD_ROOT=choose_offload_root()
WORKER_CONTRACT={
    'target':CFG.target,'id_col':CFG.id_col,'checkpoint':str(CKPT),
    'batch_size':CFG.estimator_batch_size,'use_amp':CFG.use_amp,
    'offload_mode':CFG.offload_mode,'offload_root':str(OFFLOAD_ROOT),
    'n_jobs':CFG.n_jobs_per_worker,'runtime_probe_rows':CFG.runtime_probe_rows
}
json.dump(WORKER_CONTRACT,open(OUT/'worker_contract.json','w'),indent=2)

def log_tail(path,n=80):
    try:
        lines=Path(path).read_text(errors='replace').splitlines()
        return '\n'.join(lines[-n:])
    except Exception as e:
        return f'<unable to read log: {e!r}>'

def launch(stage,fold,gpu,n_estimators,predict_test=False,runtime_probe=False,
           query_chunk_rows=None,safe_mode=False,wait=True):
    job=OUT/f'{stage}_fold{fold}'; job.mkdir(parents=True,exist_ok=True)
    log_path=job/'worker.log'; log=open(log_path,'w')
    qrows=int(query_chunk_rows or CFG.query_chunk_rows)
    cmd=[
        sys.executable,str(WORKER),'--out',str(OUT),'--fold',str(fold),
        '--stage',stage,'--n-estimators',str(n_estimators),
        '--query-chunk-rows',str(qrows)
    ]
    if predict_test: cmd.append('--predict-test')
    if runtime_probe: cmd.append('--runtime-probe')
    if safe_mode: cmd.append('--safe-mode')
    env=os.environ.copy()
    env['CUDA_VISIBLE_DEVICES']=str(gpu)
    env['TABLDM_CPU_THREADS']=str(CFG.n_jobs_per_worker)
    env['OMP_NUM_THREADS']=str(CFG.n_jobs_per_worker)
    env['MKL_NUM_THREADS']=str(CFG.n_jobs_per_worker)
    env['MALLOC_ARENA_MAX']='2'
    env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
    p=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env)
    spec={
        'stage':stage,'fold':fold,'gpu':gpu,'n_estimators':n_estimators,
        'predict_test':predict_test,'runtime_probe':runtime_probe,
        'query_chunk_rows':qrows,'safe_mode':safe_mode
    }
    if wait:
        rc=p.wait(); log.close()
        if rc!=0:
            print(log_tail(log_path))
            raise RuntimeError(f'{stage} fold {fold} failed rc={rc}; inspect {log_path}')
        return job
    return p,log,job,spec

def wait_jobs(jobs,label):
    failed=[]; completed=[]
    for p,fh,job,spec in jobs:
        rc=p.wait(); fh.close()
        if rc!=0:
            failed.append((spec,rc,job/'worker.log'))
        else:
            completed.append(spec)
    if failed:
        print(f'{label}: {len(failed)} parallel worker(s) failed. Log tails:')
        for spec,rc,lp in failed:
            print(f'\n--- fold={spec["fold"]} rc={rc} log={lp} ---')
            print(log_tail(lp))
    return failed,completed

def run_fold_batch(specs,label):
    """Run one worker per physical GPU at a time; retry only failed folds conservatively.

    With two visible T4s, independent folds run two-at-a-time. With one visible T4,
    the exact same fold jobs run sequentially. Scheduling never changes model/data/seed.
    """
    pending=[dict(x) for x in specs]
    wave_id=0
    while pending:
        wave_id+=1
        used=set(); wave=[]; rest=[]
        for s in pending:
            gpu=int(s['gpu'])
            if gpu not in VISIBLE_GPUS:
                raise RuntimeError(f'{label}: requested GPU {gpu}, visible GPUs={VISIBLE_GPUS}')
            if gpu in used:
                rest.append(s)
            else:
                used.add(gpu); wave.append(s)
        pending=rest

        jobs=[]
        for s in wave:
            p,fh,job,spec=launch(
                s['stage'],s['fold'],s['gpu'],s['n_estimators'],
                predict_test=s.get('predict_test',False),
                runtime_probe=s.get('runtime_probe',False),
                query_chunk_rows=CFG.query_chunk_rows,
                safe_mode=False,wait=False
            )
            jobs.append((p,fh,job,spec))
        print(f'{label}: wave {wave_id} launching '+', '.join(
            f'fold {x[3]["fold"]} -> GPU {x[3]["gpu"]}' for x in jobs
        ))
        failed,_=wait_jobs(jobs,f'{label} wave {wave_id}')
        if not failed:
            continue

        second_fail=[]
        # Retries are intentionally sequential, even when multiple GPUs are visible,
        # to minimize host/RAM/disk pressure after a worker was killed.
        for spec,old_rc,_ in failed:
            print(f'{label}: retry fold {spec["fold"]} after rc={old_rc} '
                  f'with query_chunk_rows={CFG.safe_query_chunk_rows} and conservative disk offload.')
            try:
                launch(
                    spec['stage'],spec['fold'],spec['gpu'],spec['n_estimators'],
                    predict_test=spec['predict_test'],runtime_probe=spec['runtime_probe'],
                    query_chunk_rows=CFG.safe_query_chunk_rows,safe_mode=True,wait=True
                )
            except Exception as e:
                second_fail.append((spec,e))
        if second_fail:
            raise RuntimeError(f'{label} failed even after conservative retry: {second_fail}')

def run_single_with_retry(stage,fold,gpu,n_estimators,predict_test=False,runtime_probe=False):
    try:
        return launch(
            stage,fold,gpu,n_estimators,predict_test=predict_test,
            runtime_probe=runtime_probe,query_chunk_rows=CFG.query_chunk_rows,
            safe_mode=False,wait=True
        )
    except Exception as first:
        print(f'{stage} fold {fold}: normal runtime failed ({first!r}); retrying conservative mode.')
        return launch(
            stage,fold,gpu,n_estimators,predict_test=predict_test,
            runtime_probe=runtime_probe,query_chunk_rows=CFG.safe_query_chunk_rows,
            safe_mode=True,wait=True
        )


Offload root /kaggle/temp/oqdq5_v14_tabldm_offload free GiB 1026.27


## 6. Stage R — full-context memory/runtime proof

The previous run proved that a 2,048-row query already consumed ~98% of a T4 while the full Stage-S validation query was much larger. This revision therefore validates the same full training context with bounded query-row inference. No training rows or features are subsampled.


In [7]:
RUNTIME_FOLD=0
runtime_dir=run_single_with_retry(
    'runtime',RUNTIME_FOLD,0,CFG.runtime_estimators,
    predict_test=False,runtime_probe=True
)
runtime_info=json.load(open(runtime_dir/'runtime.json'))
assert runtime_info['status']=='success' and runtime_info['train_rows']>70000 and runtime_info['features']==len(FEATURES)
assert runtime_info['valid_rows_predicted']<=CFG.runtime_probe_rows
json.dump(runtime_info,open(OUT/'runtime_proof.json','w'),indent=2)
print(json.dumps(runtime_info,indent=2))


{
  "status": "success",
  "stage": "runtime",
  "fold": 0,
  "seed": 14017,
  "n_estimators": 1,
  "train_rows": 75730,
  "valid_rows_predicted": 1792,
  "test_rows_predicted": 0,
  "query_chunk_rows": 1792,
  "valid_predict_calls": 1,
  "test_predict_calls": 0,
  "safe_mode": false,
  "features": 176,
  "nominal_features": 158,
  "runtime_sec": 44.17833924293518,
  "cuda_name": "Tesla T4",
  "cuda_total_gib": 14.56219482421875,
  "max_cuda_alloc_gib": 14.25057077407837,
  "cuda_peak_fraction": 0.9786004751411435,
  "peak_rss_gib": 2.523101806640625,
  "offload_dir": "/kaggle/temp/oqdq5_v14_tabldm_offload/runtime_fold0"
}


## 7. Stage S — preregistered two-fold scientific screen on both T4s


In [8]:
stage_s_specs=[
    {'stage':'screen','fold':f,'gpu':VISIBLE_GPUS[i % len(VISIBLE_GPUS)],'n_estimators':CFG.screen_estimators,'predict_test':False,'runtime_probe':False}
    for i,f in enumerate(CFG.screen_folds)
]
print('Stage S GPU assignment',[(x['fold'],x['gpu']) for x in stage_s_specs])
run_fold_batch(stage_s_specs,'Stage S')

def qwk(a,b): return float(cohen_kappa_score(np.asarray(a),np.asarray(b),weights='quadratic'))
def disc(s,t):
    t=np.sort(np.asarray(t,float)); s=np.asarray(s,float); return (1+(s>=t[0])+(s>=t[1])+(s>=t[2])).astype(int)
def init_thr(yy,ss):
    p=pd.Series(yy).value_counts(normalize=True).sort_index(); cum=np.cumsum([p.get(k,0.) for k in [1,2,3]]); return np.quantile(ss,np.clip(cum,.001,.999))
def fit_thr(yy,ss,maxiter=160):
    yy=np.asarray(yy,int); ss=np.asarray(ss,float); starts=[init_thr(yy,ss),np.array([1.55,2.03,2.52])]
    best=None
    for st in starts:
        def obj(t):
            t=np.sort(t)
            if np.min(np.diff(t))<.02: return 10+float(np.sum(np.maximum(.02-np.diff(t),0)))
            return -qwk(yy,disc(ss,t))
        r=minimize(obj,st,method='Nelder-Mead',options={'maxiter':maxiter,'xatol':1e-5,'fatol':1e-7})
        cand=np.sort(r.x); sc=qwk(yy,disc(ss,cand))
        if best is None or sc>best[1]: best=(cand,sc)
    return best[0]
def auc12(yy,ss):
    yy=np.asarray(yy,int); ss=np.asarray(ss,float); m=np.isin(yy,[1,2]); return float(roc_auc_score((yy[m]==2).astype(int),ss[m]))
def bauc(yy,ss,k=1): return float(roc_auc_score((np.asarray(yy)>k).astype(int),np.asarray(ss,float)))
def diag(yy,ss):
    return {'auc_1v2':auc12(yy,ss),'auc_ygt1':bauc(yy,ss,1),'rmse':float(np.sqrt(mean_squared_error(yy,ss))),'mae':float(mean_absolute_error(yy,ss)),'spearman':float(spearmanr(yy,ss)[0])}

screen_pred={}
for f in CFG.screen_folds:
    z=np.load(OUT/f'screen_fold{f}'/'valid_predictions.npz',allow_pickle=False)
    ids=z['id']; p=np.asarray(z['pred'],float); idx=pd.Index(train_ids).get_indexer(ids)
    if (idx<0).any() or not np.all(folds[idx]==f): raise RuntimeError(f'Stage-S fold {f} ID mapping failure.')
    screen_pred[f]=(idx,p)

rows=[]
for f in CFG.screen_folds:
    other=[x for x in CFG.screen_folds if x!=f][0]
    idx,p=screen_pred[f]; oidx,op=screen_pred[other]
    yy=y[idx]; oy=y[oidx]; rec=recovery_oof[idx]; orec=recovery_oof[oidx]
    t=fit_thr(oy,op,130); rt=fit_thr(oy,orec,130)
    d=diag(yy,p); rd=diag(yy,rec)
    rows.append({'fold':f,'n':len(idx),'tabldm_qwk':qwk(yy,disc(p,t)),'recovery_qwk':qwk(yy,disc(rec,rt)),'delta_qwk':qwk(yy,disc(p,t))-qwk(yy,disc(rec,rt)),
                 **{f'tabldm_{k}':v for k,v in d.items()},**{f'recovery_{k}':v for k,v in rd.items()},
                 'delta_auc_1v2':d['auc_1v2']-rd['auc_1v2'],'delta_auc_ygt1':d['auc_ygt1']-rd['auc_ygt1'],
                 'prediction_corr_vs_recovery':float(np.corrcoef(p,rec)[0,1]),'residual_corr_vs_recovery':float(np.corrcoef(p-yy,rec-yy)[0,1]),'threshold_source_fold':other})
screen=pd.DataFrame(rows).sort_values('fold'); screen.to_csv(OUT/'stage_s_metrics.csv',index=False)
mean_auc=float(screen.delta_auc_1v2.mean()); mean_q=float(screen.delta_qwk.mean()); mean_resid=float(screen.residual_corr_vs_recovery.mean())
no_cat=bool((screen.delta_auc_1v2>=CFG.stage_catastrophic_auc).all())
strong=bool(mean_auc>=CFG.stage_auc_gain or mean_q>=CFG.stage_qwk_gain)
novel=bool(mean_q>=CFG.stage_novel_qwk_floor and mean_resid<CFG.stage_novel_resid_corr and mean_auc>=CFG.stage_novel_auc_floor)
STAGE_PROMOTED=bool(no_cat and (strong or novel))
STAGE_SUMMARY={'promoted':STAGE_PROMOTED,'mean_delta_auc_1v2':mean_auc,'mean_delta_qwk':mean_q,'mean_residual_corr':mean_resid,'no_catastrophic_fold':no_cat,'strong_signal_gate':strong,'novelty_gate':novel}
json.dump(STAGE_SUMMARY,open(OUT/'stage_s_summary.json','w'),indent=2)
print(screen.to_string(index=False)); print(json.dumps(STAGE_SUMMARY,indent=2))


Stage S GPU assignment [(0, 0), (2, 1)]
Stage S: wave 1 launching fold 0 -> GPU 0, fold 2 -> GPU 1
 fold     n  tabldm_qwk  recovery_qwk  delta_qwk  tabldm_auc_1v2  tabldm_auc_ygt1  tabldm_rmse  tabldm_mae  tabldm_spearman  recovery_auc_1v2  recovery_auc_ygt1  recovery_rmse  recovery_mae  recovery_spearman  delta_auc_1v2  delta_auc_ygt1  prediction_corr_vs_recovery  residual_corr_vs_recovery  threshold_source_fold
    0 18933    0.479098      0.489179  -0.010081        0.759664         0.774804     0.517709    0.435744         0.503310          0.767312           0.782070       0.512201      0.426482           0.515551      -0.007648       -0.007266                     0.977538                   0.991599                      2
    2 18933    0.469171      0.478538  -0.009367        0.755637         0.769895     0.524377    0.437213         0.492538          0.762950           0.776861       0.519559      0.428460           0.504406      -0.007312       -0.006966                     0.9

## 8. Full five-fold OOF + test inference only after Stage-S promotion


In [9]:
FULL_AVAILABLE=False
if STAGE_PROMOTED:
    specs=[
        {'stage':'full','fold':f,'gpu':VISIBLE_GPUS[f % len(VISIBLE_GPUS)],'n_estimators':CFG.full_estimators,'predict_test':True,'runtime_probe':False}
        for f in range(CFG.n_folds)
    ]
    print('Full-CV GPU assignment',[(x['fold'],x['gpu']) for x in specs])
    run_fold_batch(specs,'Full CV')
    FULL_AVAILABLE=True
else:
    print('V14 falsified at Stage S. Full CV/test inference is intentionally skipped; Recovery control submission remains available.')


V14 falsified at Stage S. Full CV/test inference is intentionally skipped; Recovery control submission remains available.


## 9. Leakage-safe full evaluation, coarse nested blend, bootstrap, and submission export


In [10]:
FINAL={'full_available':FULL_AVAILABLE,'stage_s':STAGE_SUMMARY,'recommended_candidate':None,'recommended_submission':None}
if FULL_AVAILABLE:
    tab_oof=np.full(len(y),np.nan,float); coverage=np.zeros(len(y),np.int8); test_folds=[]
    id_index=pd.Index(train_ids)
    runtimes=[]
    for f in range(5):
        job=OUT/f'full_fold{f}'; runtimes.append(json.load(open(job/'runtime.json')))
        z=np.load(job/'valid_predictions.npz',allow_pickle=False); ids=z['id']; p=np.asarray(z['pred'],float); idx=id_index.get_indexer(ids)
        if (idx<0).any() or len(np.unique(ids))!=len(ids) or not np.all(folds[idx]==f): raise RuntimeError(f'Full fold {f} OOF contract failure')
        tab_oof[idx]=p; coverage[idx]+=1
        zt=np.load(job/'test_predictions.npz',allow_pickle=False)
        if not np.array_equal(zt['id'],test_ids): raise RuntimeError(f'Full fold {f} test ID contract failure')
        pt=np.asarray(zt['pred'],float)
        if not np.isfinite(pt).all(): raise RuntimeError(f'Non-finite test prediction fold {f}')
        test_folds.append(pt)
    if not np.all(coverage==1) or not np.isfinite(tab_oof).all(): raise RuntimeError('OOF coverage failure')
    tab_test=np.mean(np.vstack(test_folds),axis=0)
    json.dump(runtimes,open(OUT/'full_runtime.json','w'),indent=2)

    # Standalone cross-fitted calibration.
    tab_cls=np.zeros(len(y),int); tab_thr=[]
    for f in range(5):
        tr=folds!=f; va=~tr
        t=fit_thr(y[tr],tab_oof[tr]); tab_cls[va]=disc(tab_oof[va],t); tab_thr.append(t)
    if (tab_cls==0).any(): raise RuntimeError('Cross-fitted threshold coverage failure')

    # Frozen coarse blend family. Alpha is selected only on the other four folds by RMSE.
    alphas=np.array([0.,.25,.5,.75,1.]); blend=np.full(len(y),np.nan,float); alpha_rows=[]
    for f in range(5):
        tr=folds!=f; va=~tr
        rms=[np.sqrt(mean_squared_error(y[tr],(1-a)*recovery_oof[tr]+a*tab_oof[tr])) for a in alphas]
        a=float(alphas[int(np.argmin(rms))]); blend[va]=(1-a)*recovery_oof[va]+a*tab_oof[va]
        alpha_rows.append({'fold':f,'alpha':a,'train_rmse':float(min(rms))})
    blend_cls=np.zeros(len(y),int); blend_thr=[]
    for f in range(5):
        tr=folds!=f; va=~tr; t=fit_thr(y[tr],blend[tr]); blend_cls[va]=disc(blend[va],t); blend_thr.append(t)
    pd.DataFrame(alpha_rows).to_csv(OUT/'blend_fold_alphas.csv',index=False)

    fold_rows=[]
    for f in range(5):
        m=folds==f
        rq=qwk(y[m],recovery_class[m]); tq=qwk(y[m],tab_cls[m]); bq=qwk(y[m],blend_cls[m])
        fold_rows.append({'fold':f,'n':int(m.sum()),'recovery_qwk':rq,'tabldm_qwk':tq,'tabldm_delta_qwk':tq-rq,'blend_qwk':bq,'blend_delta_qwk':bq-rq,
                          'recovery_auc_1v2':auc12(y[m],recovery_oof[m]),'tabldm_auc_1v2':auc12(y[m],tab_oof[m]),'blend_auc_1v2':auc12(y[m],blend[m]),
                          'recovery_auc_ygt1':bauc(y[m],recovery_oof[m]),'tabldm_auc_ygt1':bauc(y[m],tab_oof[m]),'blend_auc_ygt1':bauc(y[m],blend[m])})
    fold_df=pd.DataFrame(fold_rows); fold_df.to_csv(OUT/'full_fold_metrics.csv',index=False)

    def evaluate(name,score,cls):
        fq=[]
        for f in range(5):
            m=folds==f; fq.append(qwk(y[m],cls[m])-qwk(y[m],recovery_class[m]))
        rng=np.random.default_rng(SEED+1414); boot=np.empty(CFG.bootstrap_reps)
        for b in range(CFG.bootstrap_reps):
            ix=rng.integers(0,len(y),len(y)); boot[b]=qwk(y[ix],cls[ix])-qwk(y[ix],recovery_class[ix])
        np.save(OUT/f'bootstrap_{name}_v14.npy',boot)
        return {'name':name,'qwk':qwk(y,cls),'recovery_qwk':qwk(y,recovery_class),'delta_qwk':qwk(y,cls)-qwk(y,recovery_class),
                'auc_1v2':auc12(y,score),'recovery_auc_1v2':auc12(y,recovery_oof),'delta_auc_1v2':auc12(y,score)-auc12(y,recovery_oof),
                'auc_ygt1':bauc(y,score),'delta_auc_ygt1':bauc(y,score)-bauc(y,recovery_oof),
                'rmse':float(np.sqrt(mean_squared_error(y,score))),'mae':float(mean_absolute_error(y,score)),'spearman':float(spearmanr(y,score)[0]),
                'prediction_corr':float(np.corrcoef(score,recovery_oof)[0,1]),'residual_corr':float(np.corrcoef(score-y,recovery_oof-y)[0,1]),
                'fold_deltas':list(map(float,fq)),'positive_folds':int(np.sum(np.asarray(fq)>=0)),'min_fold_delta':float(min(fq)),
                'bootstrap_p_gt0':float(np.mean(boot>0)),'bootstrap_median':float(np.median(boot)),'bootstrap_95':list(map(float,np.quantile(boot,[.025,.975])))}
    E_TAB=evaluate('tabldm',tab_oof,tab_cls); E_BLEND=evaluate('blend',blend,blend_cls)
    evals=[E_TAB,E_BLEND]; best=max(evals,key=lambda e:(e['qwk'],e['delta_auc_1v2']))
    pd.DataFrame([{k:v for k,v in e.items() if not isinstance(v,list)} for e in evals]).to_csv(OUT/'full_candidate_metrics.csv',index=False)

    scientific=bool(best['delta_qwk']>=CFG.scientific_qwk_gain and (best['positive_folds']>=4 or best['bootstrap_p_gt0']>=.95) and best['delta_auc_1v2']>=0 and best['delta_auc_ygt1']>=0)
    submit=bool(best['delta_qwk']>=CFG.submission_qwk_gain and best['positive_folds']>=CFG.submission_min_positive_folds and best['min_fold_delta']>=CFG.submission_min_fold and best['bootstrap_p_gt0']>=CFG.submission_boot_p and best['delta_auc_1v2']>=0)
    challenge=bool(submit and (best['delta_qwk']>=CFG.challenge_qwk_gain or best['delta_auc_1v2']>=CFG.challenge_auc_gain))

    pd.DataFrame({'id':train_ids,'life_satisfaction':y,'fold':folds,'recovery_score':recovery_oof,'recovery_class':recovery_class,'tabldm_score':tab_oof,'tabldm_class':tab_cls,'blend_score':blend,'blend_class':blend_cls}).to_csv(OUT/'oof_v14.csv',index=False)
    test_out=pd.DataFrame({'id':test_ids,'recovery_score':recovery_test,'tabldm_score':tab_test})
    for f,p in enumerate(test_folds): test_out[f'tabldm_fold{f}']=p
    test_out.to_csv(OUT/'test_scores_v14.csv',index=False)

    # Diagnostic standalone submission is always exported after a successful full run.
    standalone_t=np.median(np.vstack(tab_thr),axis=0); standalone_cls=disc(tab_test,standalone_t)
    standalone_sub=pd.DataFrame({'id':test_ids,'life_satisfaction':standalone_cls.astype(int)})
    assert len(standalone_sub)==CFG.expected_test_ids and standalone_sub.id.is_unique and standalone_sub.life_satisfaction.isin([1,2,3,4]).all()
    standalone_sub.to_csv(OUT/'submission_v14_tabldm_diagnostic.csv',index=False)

    recommended_path=None; final_alpha=None
    if submit:
        if best['name']=='tabldm':
            final_score=tab_test; final_t=standalone_t; final_alpha=1.0
        else:
            final_alpha=float(np.median([r['alpha'] for r in alpha_rows])); final_score=(1-final_alpha)*recovery_test+final_alpha*tab_test; final_t=np.median(np.vstack(blend_thr),axis=0)
        final_cls=disc(final_score,final_t); sub=pd.DataFrame({'id':test_ids,'life_satisfaction':final_cls.astype(int)})
        assert len(sub)==CFG.expected_test_ids and sub.id.is_unique and sub.life_satisfaction.isin([1,2,3,4]).all()
        recommended_path=OUT/'submission_v14_recommended.csv'; sub.to_csv(recommended_path,index=False)
        pd.DataFrame({'id':test_ids,'recovery_score':recovery_test,'tabldm_score':tab_test,'final_score':final_score,'final_class':final_cls}).to_csv(OUT/'test_predictions_v14_recommended.csv',index=False)

    FINAL.update({'tabldm':E_TAB,'blend':E_BLEND,'recommended_candidate':best['name'],'scientific_improvement':scientific,'submission_gate':submit,'challenge_gate':challenge,
                  'recommended_submission':None if recommended_path is None else str(recommended_path),'diagnostic_submission':str(OUT/'submission_v14_tabldm_diagnostic.csv'),'final_alpha':final_alpha})
    print(fold_df.to_string(index=False)); print(json.dumps(FINAL,indent=2))
else:
    FINAL.update({'status':'falsified_at_stage_s','recovery_control_submission':str(OUT/'submission_v14_recovery_control.csv')})


## 10. Final provenance manifest and artifact audit


In [11]:
FINAL.update({
    'version':'v14_xiaomi_tabldm_full_target_ordinal_regression',
    'timestamp_utc':pd.Timestamp.utcnow().isoformat(),
    'competition':'Objective Quest DATAQUEST 5.0',
    'frozen_recovery_public_lb':CFG.recovery_public_lb,
    'frozen_recovery_oof_qwk':RECOVERY_QWK,
    'config':asdict(CFG),'provenance':PROVENANCE,'data_dir':str(DATA),'historical_artifacts':STORE.source(),
    'semantic_contract':SEMANTIC,
    'runtime_hardware':{'visible_gpu_count':N_VISIBLE_GPUS,'visible_gpu_ids':list(VISIBLE_GPUS),'scheduling_mode':GPU_SCHEDULING_MODE},
    'constraints':[
        'Competition task data only; external Xiaomi weights are synthetic-SCM pretrained foundation-model weights, not respondent data.',
        'Unique respondent modeling; exact duplicate IDs are audited and collapsed before Xiaomi inference.',
        'id and household_id are excluded from Xiaomi features.',
        'Nominal survey codes are pandas categorical; audited continuous/ordinal fields remain numeric.',
        'No task-specific fine-tuning, pseudo-labeling, target-guided feature selection, or additional model family.',
        'One official TabLDM estimator uses one GPU. If two T4s are visible, isolated fold workers run two-at-a-time; if only one T4 is visible, identical jobs run sequentially. Conservative exact-fold retry handles runtime pressure.',
        'TabLDM estimator batch_size=1 batches ensemble members only; validation/test query rows are chunked through repeated official predict() calls to bound sequence memory.',
        'Stage R uses the full outer-training context and a bounded validation-query runtime probe.',
        'Stage S uses folds 0 and 2 with cross-screen thresholds and frozen Recovery comparison.',
        'Full five-fold inference runs only after Stage-S promotion.',
        'Headline QWK uses cross-fitted thresholds. Blend alpha is selected on the other four folds only.',
        'Recovery control submission is always exported; new recommended submission is created only if the preregistered full-CV submission gate passes.',
    ]
})
if FINAL.get('recommended_submission'): FINAL['recommended_submission_sha256']=sha256(FINAL['recommended_submission'])
FINAL['recovery_control_sha256']=sha256(OUT/'submission_v14_recovery_control.csv')
if FINAL.get('diagnostic_submission'): FINAL['diagnostic_submission_sha256']=sha256(FINAL['diagnostic_submission'])
json.dump(FINAL,open(OUT/'v14_summary.json','w'),indent=2,default=lambda o:o.item() if hasattr(o,'item') else str(o))
STORE.close()
files=sorted(p.name for p in OUT.iterdir() if p.is_file())
print('\nArtifacts:'); [print(' -',x) for x in files]
print('\nFINAL STATUS:', 'SUBMISSION READY' if FINAL.get('recommended_submission') else ('FULL CV COMPLETE; DIAGNOSTIC ONLY' if FULL_AVAILABLE else 'FALSIFIED AT STAGE S'))
print('Recovery control:',OUT/'submission_v14_recovery_control.csv')
print('Recommended:',FINAL.get('recommended_submission'))
print('Summary:',OUT/'v14_summary.json')



Artifacts:
 - foundation_provenance.json
 - runtime_proof.json
 - semantic_contract_v14.json
 - stage_s_metrics.csv
 - stage_s_summary.json
 - submission_v14_recovery_control.csv
 - tabldm_worker_v14.py
 - test_unique_v14.csv
 - train_unique_v14.csv
 - v14_summary.json
 - worker_contract.json

FINAL STATUS: FALSIFIED AT STAGE S
Recovery control: /kaggle/working/oqdq5_v14/submission_v14_recovery_control.csv
Recommended: None
Summary: /kaggle/working/oqdq5_v14/v14_summary.json
